# Deep Q-Networks

DQN makes Q-learning scalable by combining targets, replay, and a network.

DQN keeps the Q-learning target but replaces the table with a small neural approximator. Replay and target networks reduce the instability of bootstrapping from a moving network. Save a copy to Drive to edit.

In [ ]:

import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 11712
rng = np.random.default_rng(SEED)
ACTIONS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
ACTION_NAMES = np.array(["↑", "→", "↓", "←"])


@dataclass
class GridMDP:
    name: str
    rows: int
    cols: int
    start: int
    goals: tuple
    pits: tuple
    walls: tuple
    step_cost: float
    slip: float
    wind: tuple
    gamma: float = 0.9

    @property
    def n_states(self):
        return self.rows * self.cols

    @property
    def n_actions(self):
        return 4

    def to_pos(self, state):
        return divmod(int(state), self.cols)

    def to_state(self, row, col):
        return int(row * self.cols + col)

    def is_terminal(self, state):
        return int(state) in self.goals or int(state) in self.pits

    def in_wall(self, row, col):
        return self.to_state(row, col) in self.walls

    def move(self, state, action):
        if self.is_terminal(state):
            return int(state)
        row, col = self.to_pos(state)
        delta = ACTIONS[int(action)]
        new_row = row + int(delta[0]) + int(self.wind[0])
        new_col = col + int(delta[1]) + int(self.wind[1])
        blocked = new_row < 0 or new_row >= self.rows
        blocked = blocked or new_col < 0 or new_col >= self.cols
        if blocked:
            return int(state)
        if self.in_wall(new_row, new_col):
            return int(state)
        return self.to_state(new_row, new_col)

    def reward(self, next_state):
        if int(next_state) in self.goals:
            return 1.0
        if int(next_state) in self.pits:
            return -1.0
        return float(self.step_cost)

    def transitions(self, state, action):
        if self.is_terminal(state):
            return [(1.0, int(state), 0.0)]
        intended = int(action)
        probs = {intended: 1.0 - self.slip}
        if self.slip > 0:
            probs[(intended - 1) % 4] = probs.get((intended - 1) % 4, 0.0) + self.slip / 2.0
            probs[(intended + 1) % 4] = probs.get((intended + 1) % 4, 0.0) + self.slip / 2.0
        out = []
        for actual, prob in probs.items():
            next_state = self.move(state, actual)
            out.append((prob, next_state, self.reward(next_state)))
        return out

    def step(self, state, action, local_rng):
        options = self.transitions(state, action)
        probs = np.array([item[0] for item in options], dtype=float)
        choice = int(local_rng.choice(len(options), p=probs))
        _, next_state, reward = options[choice]
        return next_state, reward, self.is_terminal(next_state)


def make_f12_ladder():
    return [
        GridMDP("D1 two-state chain", 1, 2, 0, (1,), (), (), -0.01, 0.0, (0, 0)),
        GridMDP("D2 slippery three-state chain", 1, 3, 0, (2,), (), (), -0.02, 0.2, (0, 0)),
        GridMDP("D3 4x4 gridworld", 4, 4, 0, (15,), (5,), (), -0.03, 0.05, (0, 0)),
        GridMDP("D4 windy stochastic grid", 4, 5, 0, (19,), (7, 13), (6,), -0.04, 0.15, (0, 0)),
        GridMDP("D5 sparse 6x6 grid", 6, 6, 0, (35,), (14, 21, 28), (8, 9, 16), -0.02, 0.1, (0, 0)),
    ]


def value_iteration(env, iterations=200, tol=1e-10):
    values = np.zeros(env.n_states)
    q_values = np.zeros((env.n_states, env.n_actions))
    for _ in range(iterations):
        old = values.copy()
        for state in range(env.n_states):
            for action in range(env.n_actions):
                total = 0.0
                for prob, next_state, reward in env.transitions(state, action):
                    total += prob * (reward + env.gamma * old[next_state])
                q_values[state, action] = total
            values[state] = np.max(q_values[state])
            if env.is_terminal(state):
                values[state] = 0.0
        if np.max(np.abs(values - old)) < tol:
            break
    for state in range(env.n_states):
        for action in range(env.n_actions):
            total = 0.0
            for prob, next_state, reward in env.transitions(state, action):
                total += prob * (reward + env.gamma * values[next_state])
            q_values[state, action] = total
    return values, q_values


def epsilon_greedy(q_values, state, epsilon, local_rng):
    if local_rng.random() < epsilon:
        return int(local_rng.integers(q_values.shape[1]))
    return int(np.argmax(q_values[state]))


def run_episode_return(env, q_values, epsilon, local_rng, max_steps=80):
    state = env.start
    total = 0.0
    discount = 1.0
    for _ in range(max_steps):
        action = epsilon_greedy(q_values, state, epsilon, local_rng)
        next_state, reward, done = env.step(state, action, local_rng)
        total += discount * reward
        discount *= env.gamma
        state = next_state
        if done:
            break
    return total


def state_features(env, state, mode="normalized"):
    row, col = env.to_pos(state)
    if mode == "one_hot":
        x = np.zeros(env.n_states)
        x[state] = 1.0
        return x
    if mode == "raw":
        return np.array([1.0, float(row), float(col), float(row * col)])
    row_scale = max(env.rows - 1, 1)
    col_scale = max(env.cols - 1, 1)
    return np.array([1.0, row / row_scale, col / col_scale, row * col / max(row_scale * col_scale, 1)])


def plot_value_and_curve(ladder, artifacts, curve, ylabel, title):
    fig, axes = plt.subplots(2, len(ladder), figsize=(3 * len(ladder), 5))
    for idx, env in enumerate(ladder):
        values = artifacts[idx].reshape(env.rows, env.cols)
        axes[0, idx].imshow(values, cmap="viridis")
        axes[0, idx].set_title(env.name)
        axes[0, idx].set_xticks([])
        axes[0, idx].set_yticks([])
        axes[1, idx].plot(curve[idx])
        axes[1, idx].set_xlabel("episode")
        axes[1, idx].set_ylabel(ylabel)
    fig.suptitle(title)
    plt.tight_layout()


def print_ladder_preview(ladder):
    rows = []
    for name_idx, env in enumerate(ladder, start=1):
        rows.append((f"D{name_idx}", env.name, env.n_states, env.n_actions, env.start, env.goals))
    for row in rows:
        print(row)
    sample = ladder[-1]
    print("D5 sample transition", sample.transitions(sample.start, 1))


## The concept, built once (D1)

The lesson keeps the Bellman language visible:

$$V^\pi(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)\bigl(R(s,a,s')+\gamma V^\pi(s')\bigr)$$

The worked numbers used throughout Part 11 are $G=1+0.9\cdot0+0.9^2\cdot2=2.620$, $y=1+0.9\cdot0.8=1.720$, and $Q_{new}=0.4+0.5(1.720-0.4)=1.060$.


In [ ]:

gamma = 0.9
q_old = 0.4
reward = 1.0
next_q_values = np.array([0.8, 0.2, 0.1, -0.1])
q_target = reward + gamma * np.max(next_q_values)
loss_before = 0.5 * (q_old - q_target) ** 2
q_after_demo = q_old + 0.5 * (q_target - q_old)
assert np.isclose(q_target, 1.720)
assert np.isclose(q_after_demo, 1.060)
assert np.isclose(loss_before, 0.8712)
print(q_target, q_after_demo, loss_before)


Now implement a tiny NumPy DQN: one hidden ReLU layer, replay, target network, and Bellman targets.

In [ ]:

def init_tiny_network(env, hidden=8, seed=0):
    local_rng = np.random.default_rng(seed)
    input_dim = env.n_states
    w1 = local_rng.normal(0.0, 0.08, size=(input_dim, hidden))
    b1 = np.zeros(hidden)
    w2 = local_rng.normal(0.0, 0.08, size=(hidden, env.n_actions))
    b2 = np.zeros(env.n_actions)
    return {"w1": w1, "b1": b1, "w2": w2, "b2": b2}


def one_hot(env, state):
    x = np.zeros(env.n_states)
    x[state] = 1.0
    return x


def dqn_forward(params, x):
    z1 = x @ params["w1"] + params["b1"]
    h1 = np.maximum(z1, 0.0)
    q_values = h1 @ params["w2"] + params["b2"]
    return z1, h1, q_values


def dqn_update(params, target_params, env, batch, alpha=0.03, gamma=0.9, double=False):
    for state, action, reward, next_state, done in batch:
        x = one_hot(env, state)
        z1, h1, q_values = dqn_forward(params, x)
        _, _, next_online = dqn_forward(params, one_hot(env, next_state))
        _, _, next_target = dqn_forward(target_params, one_hot(env, next_state))
        if double:
            next_action = int(np.argmax(next_online))
            bootstrap = next_target[next_action]
        else:
            bootstrap = np.max(next_target)
        target = reward + gamma * bootstrap * (not done)
        error = q_values[action] - target
        grad_q = np.zeros(env.n_actions)
        grad_q[action] = error
        grad_w2 = np.outer(h1, grad_q)
        grad_b2 = grad_q
        grad_h = params["w2"] @ grad_q
        grad_z = grad_h * (z1 > 0)
        grad_w1 = np.outer(x, grad_z)
        grad_b1 = grad_z
        params["w2"] -= alpha * grad_w2
        params["b2"] -= alpha * grad_b2
        params["w1"] -= alpha * grad_w1
        params["b1"] -= alpha * grad_b1
    return params


def tiny_dqn(env, episodes=60, epsilon=0.2, seed=0, use_replay=True, use_target=True, double=False):
    local_rng = np.random.default_rng(seed)
    params = init_tiny_network(env, seed=seed)
    target_params = {key: value.copy() for key, value in params.items()}
    replay = []
    returns = []
    for episode in range(episodes):
        state = env.start
        total = 0.0
        discount = 1.0
        for _ in range(60):
            _, _, q_values = dqn_forward(params, one_hot(env, state))
            if local_rng.random() < epsilon:
                action = int(local_rng.integers(env.n_actions))
            else:
                action = int(np.argmax(q_values))
            next_state, reward, done = env.step(state, action, local_rng)
            transition = (state, action, reward, next_state, done)
            replay.append(transition)
            if len(replay) > 100:
                replay.pop(0)
            if use_replay and len(replay) >= 8:
                idx = local_rng.choice(len(replay), size=8, replace=False)
                batch = [replay[int(i)] for i in idx]
            else:
                batch = [transition]
            active_target = target_params if use_target else params
            params = dqn_update(params, active_target, env, batch, double=double)
            total += discount * reward
            discount *= env.gamma
            state = next_state
            if done:
                break
        if use_target and episode % 10 == 0:
            target_params = {key: value.copy() for key, value in params.items()}
        returns.append(total)
    values = np.zeros(env.n_states)
    for state in range(env.n_states):
        _, _, q_values = dqn_forward(params, one_hot(env, state))
        values[state] = np.max(q_values)
    return params, values, returns


## The dataset ladder
We build the F12 D1–D5 environment ladder inline: a two-state chain, a slippery chain, a 4x4 grid, a windy stochastic grid, and a larger sparse-reward grid.

In [ ]:

ladder = make_f12_ladder()
print_ladder_preview(ladder)


## Run the same method across D1–D5
Metric: return. The loops are tiny, seeded, and CPU-only.

In [ ]:

ladder = make_f12_ladder()
metrics = []
artifacts = []
curves = []
for idx, env in enumerate(ladder):
    params, values, returns = tiny_dqn(env, episodes=40, seed=SEED + idx, use_replay=True, use_target=True)
    metrics.append(np.mean(returns[-10:]))
    artifacts.append(values)
    curves.append(returns)
print("rung | metric=return")
for idx, value in enumerate(metrics, start=1):
    print(f"D{idx}: {value:.4f}")


## Results visualization
The closing figure shows value/policy heatmaps for each rung and a learning curve for the same metric.

In [ ]:

plot_value_and_curve(ladder, artifacts, curves, "return", "Tiny NumPy DQN: value heatmaps and return curves")


### Pitfall on D5: bootstrapping from a moving target
A DQN without replay and a target network updates on correlated transitions while chasing its own changing prediction.

In [ ]:

env = make_f12_ladder()[-1]
_, _, unstable = tiny_dqn(env, episodes=25, seed=5, use_replay=False, use_target=False)
_, _, stable = tiny_dqn(env, episodes=25, seed=5, use_replay=True, use_target=True)
print("without replay/target mean return", round(float(np.mean(unstable[-5:])), 3))
print("with replay/target mean return", round(float(np.mean(stable[-5:])), 3))


## Evaluate it + Practice
- Compare the metric against a no-skill random-action baseline before trusting the learned policy.
- Sanity check D1 first: the backed-up value should move toward the exact worked target.
- Ablate the key idea for this lesson and confirm the metric gets worse or less stable.
- Failure signals: exploding values, flat returns, unvisited actions, or mismatched table shapes.

Practice prompts:
1. Change the discount from 0.9 to 0.7 and predict which rungs lose the most value.


2. Add one extra pit to D5 and rerun the same metric table.

3. Replace the policy heatmap with arrows from `ACTION_NAMES[np.argmax(Q, axis=1)]` where Q is available.